# [1.2] Induction circuits: 현대화된 TransformerLens 실습

**Ubuntu + uv + TransformerLens 3.6 (2026-08)**

이 노트북은 ARENA의 induction-circuit 실습을 2026년의 TransformerLens API와 연구 맥락으로 다시 쓴 버전이다. GPT-2와 2-layer attention-only toy model을 이용해 **관찰 → 정량화 → 직접 기여 → 개입 → weight 가설 → path 검증**의 실험 문법을 연습한다.

> 핵심 위치: induction circuit은 일반 ICL의 완성 이론이 아니라, 완전히 해부 가능한 **token-level associative copying**의 표준 예제다.

## 2026년 관점

고전 induction algorithm은 `[A, B, …, A] → B`이다. 이전 token 정보를 한 위치 뒤에 쓰는 head와, 현재 token과 그 정보를 matching해 continuation을 읽는 head가 결합한다. 이 노트북에서 반복 random token의 후반 loss가 내려가는 현상은 **exact in-context copying / repetition exploitation**의 행동 증거다. 분류·번역·함수 학습 같은 abstractive few-shot ICL 전체의 증거로 읽지 않는다.

현대적 판정 순서는 다음이다.

`pattern metric → OV/direct-logit function → causal intervention → path-specific control → 여러 prompt에서 재현`

따라서 높은 induction score는 `induction-head candidate`를 뜻할 뿐이다. candidate가 정답 logit을 올리는지, 실제 task에 필요한지, 추정한 edge를 통해 효과를 내는지는 별도 실험으로 확인한다.

## 학습 목표

1. TransformerLens 3의 `TransformerBridge`로 최신 HuggingFace 모델을 읽고 cache한다.
2. attention pattern을 detector로 바꾸고, detector와 기능을 구분한다.
3. direct logit attribution(DLA), ablation, activation patching의 주장 범위를 구분한다.
4. attention-only toy model에서 QK/OV, residual decomposition, `FactoredMatrix`, composition을 계산한다.
5. 그럴듯한 circuit story가 아니라 task-specific causal evidence를 기준으로 결론을 낸다.

## Ubuntu + uv 환경

GTX 1070은 Pascal(sm_61) GPU다. 2026년 PyPI 기본 PyTorch wheel은 CUDA 13 계열이며 Pascal을 지원하지 않는다. 이 노트북을 GTX 1070에서 GPU로 실행하려면 **CUDA 12.6 PyTorch index**를 명시한다. FP32를 사용한다. Pascal에는 Tensor Core가 없으므로 FP16 이득이 작고, 이 분석은 수치 해석이 중요하다.

```bash
mkdir -p ~/mech-interp && cd ~/mech-interp
uv init --bare --python 3.12
uv add --index pytorch-cu126=https://download.pytorch.org/whl/cu126 torch
uv add "transformer-lens==3.6.0" "circuitsvis==1.43.3" "plotly>=6" \
       "ipykernel>=7" "jupyterlab>=4"
uv run python -m ipykernel install --user --name tl-3-6 --display-name "Python (TransformerLens 3.6)"
uv run jupyter lab
```

JupyterLab에서 **Python (TransformerLens 3.6)** 커널을 선택한다. `uv.lock`은 유지한다. GPU가 없거나 CUDA가 인식되지 않으면 코드가 CPU로 fallback하지만, all-head ablation은 느릴 수 있다.

In [1]:
from __future__ import annotations

import functools
import inspect
from importlib.metadata import version

import circuitsvis as cv
import numpy as np
import plotly.express as px
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from packaging.version import Version
from transformer_lens import FactoredMatrix, HookedTransformer, HookedTransformerConfig
from transformer_lens import utilities
from transformer_lens.model_bridge import TransformerBridge

assert Version(version("transformer-lens")) == Version("3.6.0"), \
    "Run: uv add transformer-lens==3.6.0"
assert Version(version("transformers")) >= Version("5.4.0")
assert {"model_name", "load_weights"} <= set(inspect.signature(TransformerBridge.boot_transformers).parameters)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
torch.set_grad_enabled(False)
print(f"Torch: {torch.__version__}")
print(f"TransformerLens: {version('transformer-lens')}")
print(f"Transformers: {version('transformers')}")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
    assert torch.cuda.get_device_capability(0)[0] >= 6, "This notebook assumes a CUDA-capable GPU."
else:
    print("CPU fallback: use shorter sequences and skip exhaustive ablations if needed.")

/home/ldg/workspace/ARENA_3.0/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.13.0+cu130
TransformerLens: 3.6.0
Transformers: 5.15.0
Device: cpu
CPU fallback: use shorter sequences and skip exhaustive ablations if needed.


In [2]:
def to_numpy(x: torch.Tensor | np.ndarray) -> np.ndarray:
    return x.detach().float().cpu().numpy() if isinstance(x, torch.Tensor) else x


def imshow(x, title: str, x_labels=None, y_labels=None, **layout):
    fig = px.imshow(to_numpy(x), aspect="auto", color_continuous_scale="RdBu")
    fig.update_layout(title=title, xaxis_title="source / head", yaxis_title="destination / layer", **layout)
    if x_labels is not None:
        fig.update_xaxes(tickmode="array", tickvals=list(range(len(x_labels))), ticktext=x_labels)
    if y_labels is not None:
        fig.update_yaxes(tickmode="array", tickvals=list(range(len(y_labels))), ticktext=y_labels)
    fig.show()
    return fig


def line(y, title: str, x=None, **layout):
    fig = px.line(x=x, y=to_numpy(y))
    fig.update_layout(title=title, xaxis_title="position", yaxis_title="value", **layout)
    fig.show()
    return fig


def correct_log_probs(logits: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
    """log p(tokens[:, s+1] | tokens[:, :s+1]) for every position s."""
    return logits[:, :-1].log_softmax(-1).gather(-1, tokens[:, 1:].unsqueeze(-1)).squeeze(-1)


def diagonal_mean(pattern: torch.Tensor, offset: int) -> torch.Tensor:
    """Mean attention on source = destination + offset; preserves batch/head dims."""
    return pattern.diagonal(offset=offset, dim1=-2, dim2=-1).mean(dim=-1)

# 1️⃣ TransformerLens: Introduction

TransformerLens 3의 새 코드에서는 `HookedTransformer.from_pretrained` 대신 `TransformerBridge.boot_transformers`를 사용한다. Bridge는 HuggingFace의 원래 구현을 감싸고 canonical hook을 제공한다. `HookedTransformer`는 아래의 오래된 toy checkpoint를 읽는 호환성 경계에서만 사용한다.

`enable_compatibility_mode()`는 legacy TransformerLens 수치(접힌 LayerNorm, centered weights)에 맞출 때만 쓴다. 여기서는 고전 교재와 비교하기 위해 켠다. 새 분석을 시작할 때는 raw HF weights를 유지한 기본 Bridge부터 보는 편이 낫다.

In [3]:
GPT2_ID = "openai-community/gpt2"

# First run downloads GPT-2 Small (~124M parameters) from HuggingFace.
gpt2 = TransformerBridge.boot_transformers(GPT2_ID, device=device, dtype=torch.float32)
gpt2.enable_compatibility_mode(disable_warnings=True) # 이 부분이 LayerNorm 같은걸 없애는 듯 함

print(type(gpt2).__name__)
print({name: getattr(gpt2.cfg, name) for name in ["n_layers", "n_heads", "d_model", "d_vocab", "n_ctx"]})

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2042.71it/s]


TransformerBridge
{'n_layers': 12, 'n_heads': 12, 'd_model': 768, 'd_vocab': 50257, 'n_ctx': 1024}


In [4]:
model_description_text = (
    "TransformerLens exposes hooks on a language model so we can inspect and intervene on "
    "the computations that produce each next-token prediction."
)
model_description_text = (
"An expanding sofic approximation of G would force its distinguished subgroup J to be LEF, contradicting the choice of J as a non-LEF group."
)
with torch.inference_mode():
    loss = gpt2(model_description_text, return_type="loss")
print(f"GPT-2 loss: {loss.item():.3f}")

tokens = gpt2.to_tokens("gpt2")
print("token ids:", tokens.tolist())
print("token strings:", gpt2.to_str_tokens("gpt2"))
print("round trip:", gpt2.to_string(tokens))

GPT-2 loss: 5.134
token ids: [[50256, 70, 457, 17]]
token strings: ['<|endoftext|>', 'g', 'pt', '2']
round trip: ['<|endoftext|>gpt2']


### Detector를 mechanism 판정기로 오해하지 않기

offset diagonal 평균은 훌륭한 detector다. 하지만 높은 score가 있어도 correct token을 억제하는 anti-induction-like head, redundant head, 다른 task에 무관한 head가 있을 수 있다. 다음 장에서 DLA와 ablation을 붙여 `pattern → function → causal effect`를 검사한다.

또한 token-level diagonal OV score가 낮아도 multi-token lexical / semantic continuation을 운반하는 head일 수 있다. 이 노트북의 OV test는 exact token copying에 한정된 probe다.

# 3️⃣ Hooks, direct logit attribution, intervention

Residual stream이 additive일 때 component가 최종 정답 logit에 직접 쓴 양은 계산할 수 있다.

`DLA(component, correct token) = component · W_U[:, correct token]`

DLA는 **direct logit attribution**이다. downstream Q/K/V 변화, LayerNorm 재조정, nonlinear MLP, backup circuit은 측정하지 않는다. 따라서 DLA가 작아도 ablation effect가 클 수 있다.

In [10]:
from huggingface_hub import hf_hub_download


cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
    device="cpu"
)
REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

toy = HookedTransformer(cfg)
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
toy.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [11]:
demo_text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

In [12]:
demo_tokens = toy.to_tokens(demo_text)
demo_logits, demo_cache = toy.run_with_cache(demo_tokens, remove_batch_dim=True)

In [13]:
print(type(demo_cache))

<class 'transformer_lens.ActivationCache.ActivationCache'>


In [39]:
# pattern 이 키, 0이 몇 번째 Layer 이냐이다
demo_cache['pattern', 0].shape

torch.Size([12, 62, 62])

In [22]:
dir(demo_cache)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_get_cached_ln_scale',
 '_over_ssm_layers',
 '_stack_neuron_results_apply_ln_projected',
 'accumulated_resid',
 'apply_ln_to_stack',
 'apply_slice_to_batch_dim',
 'cache_dict',
 'compute_head_results',
 'compute_ssm_effective_attention',
 'compute_ssm_state',
 'decompose_resid',
 'get_full_resid_decomposition',
 'get_neuron_results',
 'has_batch_dim',
 'has_embed',
 'has_pos_embed',
 'items',
 'keys',
 'logit_attrs',
 'model',
 'remove_batch_dim',
 'ssm_layers',
 'stack_activation',
 'stack_head_results',
 'stack_neuron_results',
 'to',
 'toggle_autodiff',
 'value

In [45]:
#(Batch 가 없으므로) Query: 62토큰, 12헤드, 64차원
embed_dim = demo_cache['k', 0][-2]
demo_cache['k', 0].shape


torch.Size([62, 12, 64])

In [ ]:
# Excercise: Attention 계산
qk = torch.einsum("qhd, khd -> hqk", demo_cache['q',0], demo_cache['k',0])


In [62]:
import math
logits = qk/math.sqrt(toy.cfg.d_head)

In [64]:
demo_cache["pattern", 0].shape

torch.Size([12, 62, 62])

In [66]:
seq_len = logits.size(-1)

future_mask = torch.triu(
    torch.ones(seq_len, seq_len, dtype=torch.bool, device=logits.device),
    diagonal=1,
)

masked_logits = logits.masked_fill(
    future_mask,
    torch.finfo(logits.dtype).min,
)

attn_weight = masked_logits.softmax(dim=-1)

In [68]:
expected = demo_cache["pattern", 0]

print((expected - attn_weight).abs().max())
torch.testing.assert_close(expected, attn_weight, rtol=1e-4, atol=1e-5)
# 마스크 전: 미래 token에도 attention을 둠

tensor(0.)


In [21]:

attention_pattern = demo_cache["pattern", 0]
print(attention_pattern.shape)
toy_tokens = toy.to_str_tokens(demo_text)

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_patterns(
        tokens=toy_tokens,
        attention=attention_pattern,
        # attention_head_names=[f"L0H{i}" for i in range(12)],
    )
)

torch.Size([12, 62, 62])
Layer 0 Head Attention Patterns:


In [ ]:
def logit_attribution(embed, layer0_result, layer1_result, W_U, tokens):
    """Direct correct-token attribution: [position, direct + 12 L0 + 12 L1]."""
    correct_unembed = W_U[:, tokens[1:]]
    direct = torch.einsum("sd,ds->s", embed[:-1], correct_unembed)
    l0 = torch.einsum("shd,ds->sh", layer0_result[:-1], correct_unembed)
    l1 = torch.einsum("shd,ds->sh", layer1_result[:-1], correct_unembed)
    return torch.cat((direct[:, None], l0, l1), dim=-1)



dla = logit_attribution(
    demo_cache["embed"], demo_cache["result", 0], demo_cache["result", 1], toy.W_U, demo_tokens[0]
)
# TransformerLens 3 keeps the batch dimension in logits, even when the cache is unbatched.
correct_logits = demo_logits[0, :-1].gather(-1, demo_tokens[0, 1:].unsqueeze(-1)).squeeze(-1)
torch.testing.assert_close(dla.sum(-1), correct_logits, atol=1e-3, rtol=0)
imshow(dla.T, "Direct logit attribution: direct path, L0 heads, L1 heads")

<class 'transformer_lens.ActivationCache.ActivationCache'>
torch.Size([12, 15, 15])
Layer 0 Head Attention Patterns:


In [ ]:
toy_mode

In [13]:
demo_text = "Mechanistic interpretability studies the computation implemented by a trained model."
demo_tokens = gpt2.to_tokens(demo_text)

In [19]:
demo_cache

ActivationCache with keys ['embed.hook_in', 'hook_embed', 'embed.hook_out', 'pos_embed.hook_in', 'hook_pos_embed', 'pos_embed.hook_out', 'blocks.0.hook_in', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_in', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.ln1.hook_out', 'blocks.0.attn.hook_in', 'blocks.0.attn.q.hook_in', 'blocks.0.attn.hook_q', 'blocks.0.attn.q.hook_out', 'blocks.0.attn.k.hook_in', 'blocks.0.attn.hook_k', 'blocks.0.attn.k.hook_out', 'blocks.0.attn.v.hook_in', 'blocks.0.attn.hook_v', 'blocks.0.attn.v.hook_out', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.attn.o.hook_in', 'blocks.0.attn.o.hook_out', 'blocks.0.attn.hook_hidden_states', 'blocks.0.hook_attn_out', 'blocks.0.attn.hook_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_in', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.ln2.hook_out', 'blocks.0.mlp.hook_in', 'blocks.0.mlp.in.hook_in', 'blocks.0.mlp.hook_pre', 'blo

In [23]:
demo_cache['blocks.0.attn.hook_q'].shape

torch.Size([15, 12, 64])

In [28]:
demo_cache['blocks.0.attn.hook_pattern'].shape

torch.Size([12, 15, 15])

In [ ]:
def zero_head_hook(z: torch.Tensor, hook, head: int):
    z[:, :, head, :] = 0.0
    return z


def head_ablation_scores(model: HookedTransformer, tokens: torch.Tensor) -> torch.Tensor:
    """Loss increase on the repeated-token second half after zeroing each head output."""
    seq_len = (tokens.shape[1] - 1) // 2
    baseline = -correct_log_probs(model(tokens), tokens)[:, -(seq_len - 1):].mean()
    result = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=device)
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            logits = model.run_with_hooks(
                tokens,
                fwd_hooks=[(utilities.get_act_name("z", layer), functools.partial(zero_head_hook, head=head))],
            )
            loss = -correct_log_probs(logits, tokens)[:, -(seq_len - 1):].mean()
            result[layer, head] = loss - baseline
    return result


# 144 forward passes. On GTX 1070 this is usually a time cost, not a VRAM cost.
zero_ablation = head_ablation_scores(toy, rep_tokens)
imshow(zero_ablation, "Zero-ablation loss increase (second half)")

triage = []
for layer in range(toy.cfg.n_layers):
    for head in range(toy.cfg.n_heads):
        triage.append({
            "head": f"{layer}.{head}",
            "prefix_score": round(scores["induction"][layer, head].item(), 3),
            "zero_ablation_loss": round(zero_ablation[layer, head].item(), 3),
        })
print(*sorted(triage, key=lambda row: row["prefix_score"], reverse=True)[:8], sep="\n")

In [ ]:
def mean_ablation_scores(model: HookedTransformer, tokens: torch.Tensor, reference_tokens: torch.Tensor) -> torch.Tensor:
    """Distributional mean ablation; less OOD than zero but not a neutral intervention."""
    _, reference_cache = model.run_with_cache(reference_tokens, names_filter=lambda n: n.endswith("hook_z"))
    mean_z = [reference_cache["z", layer].mean(dim=(0, 1)) for layer in range(model.cfg.n_layers)]
    seq_len = (tokens.shape[1] - 1) // 2
    baseline = -correct_log_probs(model(tokens), tokens)[:, -(seq_len - 1):].mean()
    result = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=device)
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            def replace_with_mean(z, hook, head=head, layer=layer):
                z[:, :, head, :] = mean_z[layer][head]
                return z
            logits = model.run_with_hooks(tokens, fwd_hooks=[(utilities.get_act_name("z", layer), replace_with_mean)])
            result[layer, head] = -correct_log_probs(logits, tokens)[:, -(seq_len - 1):].mean() - baseline
    return result


reference_tokens = generate_repeated_tokens(toy, SEQ_LEN, batch_size=16)
mean_ablation = mean_ablation_scores(toy, rep_tokens, reference_tokens)
imshow(mean_ablation, "Mean-ablation loss increase")

# A small effect is compatible with redundancy or self-repair; a large effect removes every path through that head.

### Clean / corrupted activation patching

Ablation asks whether a component is necessary under one intervention. Patching asks whether information from a clean run is sufficient to restore a chosen behavior in a corrupted run. Always define a task metric and include controls (random same-layer head, matched position, reverse patch). The example below patches one induction head's **key activation**, which is more specific than ablating the whole head but is still not a complete edge-level proof.

In [ ]:
def patch_one_head(clean_activation: torch.Tensor, head: int):
    def patch_hook(activation: torch.Tensor, hook):
        activation[:, :, head, :] = clean_activation[:, :, head, :]
        return activation
    return patch_hook


clean_tokens = generate_repeated_tokens(toy, SEQ_LEN)
corrupted_tokens = clean_tokens.clone()
# Break one earlier token identity while preserving sequence length and all later positions.
corrupted_tokens[:, 8] = (corrupted_tokens[:, 8] + 1) % toy.cfg.d_vocab

clean_logits, clean_cache = toy.run_with_cache(clean_tokens, names_filter=lambda n: n == utilities.get_act_name("k", 1))
corrupted_logits = toy(corrupted_tokens)
patched_logits = toy.run_with_hooks(
    corrupted_tokens,
    fwd_hooks=[(utilities.get_act_name("k", 1), patch_one_head(clean_cache["k", 1], head=4))],
)

position = SEQ_LEN + 8  # this logit predicts the following repeated token
correct_token = clean_tokens[0, position + 1].item()
def correct_logit(logits): return logits[0, position, correct_token].item()
print({"clean_correct_logit": correct_logit(clean_logits), "corrupted_correct_logit": correct_logit(corrupted_logits), "patched_key_correct_logit": correct_logit(patched_logits)})
# Interpret only relative to explicit controls and many random draws, not this single prompt.

# 4️⃣ Reverse-engineering the induction circuit

이 toy model에서는 weight product가 강력하다. attention-only, no LayerNorm, no MLP라는 조건 덕분에 `W_E W_V W_O W_U`와 같은 선형식을 실제 계산과 직접 연결할 수 있다.

일반 LLM에서는 pre-norm, MLP, gating, RoPE, GQA, polysemantic head 때문에 raw `W_Q W_Kᵀ`가 완전한 effective circuit이 아니다. 대형 모델의 weight product는 **가능한 선형 path** 가설이지 사용 증거가 아니다.

In [ ]:
# FactoredMatrix avoids materializing a vocab × vocab matrix (~50k² entries).
A = torch.randn(5, 2, device=device)
B = torch.randn(2, 5, device=device)
AB = FactoredMatrix(A, B)
print("shape / factor dimensions:", AB.shape, AB.ldim, AB.mdim, AB.rdim)
print("factorized norm agrees with dense norm:", AB.norm().item(), (A @ B).norm().item())


def full_ov_circuit(model: HookedTransformer, layer: int, head: int) -> FactoredMatrix:
    # source token → attended token's output-logit effect
    return FactoredMatrix(
        model.W_E @ model.W_V[layer, head],
        model.W_O[layer, head] @ model.W_U,
    )


def top1_copy_accuracy(circuit: FactoredMatrix, vocab_size: int, batch_size: int = 512) -> float:
    indices = torch.randint(0, vocab_size, (batch_size,), device=device)
    sampled_rows = circuit[indices].AB  # [sampled source token, candidate output token]
    return (sampled_rows.argmax(dim=-1) == indices).float().mean().item()

ov_14 = full_ov_circuit(toy, layer=1, head=4)
ov_110 = full_ov_circuit(toy, layer=1, head=10)
print("L1H4 top-1 exact-copy accuracy:", top1_copy_accuracy(ov_14, toy.cfg.d_vocab))
print("L1H10 top-1 exact-copy accuracy:", top1_copy_accuracy(ov_110, toy.cfg.d_vocab))

# Two heads can jointly implement a more functional circuit than either head alone.
effective_ov = FactoredMatrix(
    toy.W_E @ torch.cat((toy.W_V[1, 4], toy.W_V[1, 10]), dim=-1),
    torch.cat((toy.W_O[1, 4] @ toy.W_U, toy.W_O[1, 10] @ toy.W_U), dim=0),
)
print("combined top-1 exact-copy accuracy:", top1_copy_accuracy(effective_ov, toy.cfg.d_vocab))

In [ ]:
# Previous-token position preference: this Shortformer toy model has additive W_pos.
prev_head = 7
W_QK_prev = toy.W_Q[0, prev_head] @ toy.W_K[0, prev_head].T
pos_scores = toy.W_pos @ W_QK_prev @ toy.W_pos.T / toy.cfg.d_head**0.5
mask = torch.tril(torch.ones_like(pos_scores), diagonal=0).bool()
pos_pattern = torch.where(mask, pos_scores, torch.finfo(pos_scores.dtype).min).softmax(-1)
print("mean attention to previous position:", pos_pattern.diag(-1).mean().item())
imshow(pos_pattern[:128, :128], "L0H7 positional attention pattern (Shortformer only)")

# Do not carry this additive token/position decomposition unchanged to RoPE models:
# q_i = R_i W_Q x_i, k_j = R_j W_K x_j, so position is encoded in R_i^T R_j instead.

In [ ]:
def decompose_layer1_input(cache) -> torch.Tensor:
    """[embed, pos_embed, each L0 head result] at each position; batch dimension removed."""
    l0_results = cache["result", 0].permute(1, 0, 2)  # [head, position, d_model]
    return torch.cat((cache["embed"][None], cache["pos_embed"][None], l0_results), dim=0)


def decompose_q_or_k(components: torch.Tensor, model: HookedTransformer, head: int, which: str) -> torch.Tensor:
    W = model.W_Q[1, head] if which == "q" else model.W_K[1, head]
    return torch.einsum("cpm,md->cpd", components, W)


def decompose_attention_scores(q_parts: torch.Tensor, k_parts: torch.Tensor) -> torch.Tensor:
    return torch.einsum("aqd,bkd->abqk", q_parts, k_parts) / toy.cfg.d_head**0.5

_, _, single_cache = run_repeated(toy, SEQ_LEN)
single_cache.remove_batch_dim()
components = decompose_layer1_input(single_cache)
q_parts = decompose_q_or_k(components, toy, head=4, which="q")
k_parts = decompose_q_or_k(components, toy, head=4, which="k")
torch.testing.assert_close(q_parts.sum(0), single_cache["q", 1][:, 4], rtol=2e-2, atol=1e-3)
torch.testing.assert_close(k_parts.sum(0), single_cache["k", 1][:, 4], rtol=2e-2, atol=1e-3)
labels = ["embed", "pos_embed"] + [f"0.{h}" for h in range(toy.cfg.n_heads)]
imshow(q_parts.pow(2).sum(-1), "L1H4 query-component norms", y_labels=labels)
imshow(k_parts.pow(2).sum(-1), "L1H4 key-component norms", y_labels=labels)

In [ ]:
def k_composition_circuit(model: HookedTransformer, prev_head: int, induction_head: int) -> FactoredMatrix:
    """Current token → previous-token head write → L1 induction-head key match."""
    left = model.W_E @ model.W_Q[1, induction_head]
    right = (
        model.W_K[1, induction_head].T
        @ model.W_O[0, prev_head].T
        @ model.W_V[0, prev_head].T
        @ model.W_E.T
    )
    return FactoredMatrix(left, right)


k_comp = k_composition_circuit(toy, prev_head=7, induction_head=4)
print("K-composition top-1 same-token score:", top1_copy_accuracy(k_comp.T, toy.cfg.d_vocab))

def composition_score(W_A: torch.Tensor, W_B: torch.Tensor) -> float:
    return (W_A @ W_B).norm().div(W_A.norm() * W_B.norm()).item()

W_QK = toy.W_Q @ toy.W_K.transpose(-1, -2)
W_OV = toy.W_V @ toy.W_O
composition = {name: torch.empty((toy.cfg.n_heads, toy.cfg.n_heads), device=device) for name in ["Q", "K", "V"]}
for source_head in range(toy.cfg.n_heads):
    for destination_head in range(toy.cfg.n_heads):
        composition["Q"][source_head, destination_head] = composition_score(W_OV[0, source_head], W_QK[1, destination_head])
        composition["K"][source_head, destination_head] = composition_score(W_OV[0, source_head], W_QK[1, destination_head].T)
        composition["V"][source_head, destination_head] = composition_score(W_OV[0, source_head], W_OV[1, destination_head])
imshow(composition["K"], "Potential K-composition: L0 write → L1 read")

# This measures subspace alignment (potential connectivity), not task-relevant information flow.

In [ ]:
def batched_composition_scores(W_As: FactoredMatrix, W_Bs: FactoredMatrix) -> torch.Tensor:
    """All pair scores without materializing d_model × d_model products."""
    products = W_As[:, None] @ W_Bs[None, :]
    return products.norm() / (W_As.norm()[:, None] * W_Bs.norm()[None, :])

factored_qk = FactoredMatrix(toy.W_Q, toy.W_K.transpose(-1, -2))
factored_ov = FactoredMatrix(toy.W_V, toy.W_O)
torch.testing.assert_close(batched_composition_scores(factored_ov[0], factored_qk[1].T), composition["K"])
print("Batched FactoredMatrix composition agrees with the explicit loop.")

# Avoid `full_circuit.AB` for a full vocab × vocab circuit: it can exceed 8GB by itself.

In [ ]:
def induction_score_after_prev_head_ablation(prev_head: int | None, induction_head: int = 4) -> float:
    observed = {}
    def ablate_z(z, hook):
        if prev_head is not None:
            z[:, :, prev_head, :] = 0.0
        return z
    def read_pattern(pattern, hook):
        observed["score"] = diagonal_mean(pattern[:, induction_head], -(SEQ_LEN - 1)).mean().item()
    toy.run_with_hooks(
        rep_tokens,
        fwd_hooks=[
            (utilities.get_act_name("z", 0), ablate_z),
            (utilities.get_act_name("pattern", 1), read_pattern),
        ],
    )
    return observed["score"]

baseline = induction_score_after_prev_head_ablation(None)
changes = torch.tensor([induction_score_after_prev_head_ablation(h) - baseline for h in range(toy.cfg.n_heads)])
print("baseline L1H4 induction score:", baseline)
print("change after each L0-head ablation:", changes.tolist())
line(changes, "Effect of ablating each L0 head on L1H4's induction pattern")

# This is stronger than a correlational stripe, but still removes all L0-head output paths.
# The next refinement is to patch only the contribution of L0H7 to L1H4's key input,
# then compare Q/K/V and matched-position controls.

# 5️⃣ What this circuit does—and does not—establish

이 toy model에서는 `L0 previous-token write → L1 key match → L1 copying OV → correct logit`이라는 설명이 pattern, activation decomposition, weight product, ablation으로 함께 지지된다. 그래서 **exact token induction**에 대한 강한 mechanistic example이다.

하지만 큰 pre-norm RoPE/GQA 모델에서는 다음을 추가로 검사해야 한다.

- high prefix score가 아니라 task-specific correct-token promotion / suppression
- clean↔corrupted restoration, reverse patch, random-head·matched-position negative control
- 여러 prompt와 seed에서의 재현성
- head-level graph의 polysemanticity를 줄이는 feature/SAE-level 분석
- 큰 graph에서는 EAP/AtP류를 후보 탐색용으로 쓰고, 핵심 edge는 실제 patching으로 재검증

추천 확장 실습: (A) 모든 high-score head를 pattern·DLA·ablation 세 값으로 분류, (B) exact repetition과 multi-token/semantic copying 비교, (C) answer가 context에 없는 abstractive few-shot task를 별도 평가.

## 참고 자료

- TransformerLens 3 migration: https://transformerlensorg.github.io/TransformerLens/content/migrating_to_v3.html
- TransformerBridge model structure: https://transformerlensorg.github.io/TransformerLens/content/model_structure.html
- Olsson et al. (2022), *In-context Learning and Induction Heads*: https://arxiv.org/abs/2209.11895
- Sahin et al. (2025), *In-Context Learning Without Copying*: https://arxiv.org/abs/2511.05743
- Mohammed & Belz (2026), *What Matters More for ICL under Matched Compute Budgets?*: https://openreview.net/forum?id=c01qNs6Ew7
- uv + PyTorch index configuration: https://docs.astral.sh/uv/guides/integration/pytorch/
- PyTorch 2.13 release: CUDA 13 is the default while CUDA 12.6 is the legacy Pascal-compatible route: https://pytorch.org/blog/pytorch-2-13-release-blog/

> 마지막 원칙: 좋은 mechanistic explanation은 예쁜 attention 그림이 아니라, 사전에 정한 metric·통제군·개입에서 새로운 결과를 예측하고 재현하는 설명이다.